In [157]:
import uuid

class utils:
    def slashCheck(self, endpoint):
        if(endpoint.startswith("/")):
            return endpoint[1:]
        else:
            return endpoint
        
    def slashConverter(self, endpoint):
        return endpoint.replace("/", "\\")

    def createUuid(self):
        return str(uuid.uuid4())

    def joinPath(self, endpoint, seperator):
        endpointSplit= [p for p in endpoint.split("/")[-3:] if p] 
        return seperator.join(endpointSplit)
    
    def disablePayloadProp(self, endpoint):
        result = self.joinPath(endpoint, ".")
        return result + ".disable.payload.logs"

    def implFlowName(self,endpoint):
        return f"impl-{self.joinPath(endpoint, "-")}-subflow"
        
        

In [145]:
utilsObj = utils()

In [147]:
utilsObj.implFlowName("/c/v/a/b")

'impl-v-a-b-subflow'

In [155]:
class startFlow():
    def createMainFlow(apiName, endpoint):
        flowCode = f"""
            <flow name="post:{utilsObj.slashCheck(utilsObj.slashConverter(endpoint))}:application\\json:{apiName}-config">
                <logger level="INFO" doc:name="Start Main Process - {(endpoint)}" 
                        doc:id="{utilsObj.createUuid()}" 
                        message='#[%dw 2.0
                        output application/json
                        ---
                        if (Mule::p("{utilsObj.disablePayloadProp(endpoint)}") default false) {{}} else payload]'/>
                
                <flow-ref doc:name="{utilsObj.implFlowName(endpoint)}" 
                          doc:id="{utilsObj.createUuid()}" 
                          name="{utilsObj.implFlowName(endpoint)}"/>
                
                <logger level="INFO" doc:name="End Main Process - {(endpoint)}" 
                        doc:id="{utilsObj.createUuid()}" 
                        message='#[%dw 2.0
                        output application/json
                        ---
                        if (Mule::p("{utilsObj.disablePayloadProp(endpoint)}") default false) {{}} else payload]'/>
            </flow>
        """
        return flowCode


In [156]:
print(startFlow.createMainFlow("test-api", "/test/api"))



            <flow name="post:\test\api:application\json:test-api-config">
                <logger level="INFO" doc:name="Start Main Process - /test/api" 
                        doc:id="44be5aff-b75b-437a-ae0f-df915578352c" 
                        message='#[%dw 2.0
                        output application/json
                        ---
                        if (Mule::p("test.api.disable.payload.logs") default false) {} else payload]'/>

                <flow-ref doc:name="impl-test-api-subflow" 
                          doc:id="6e7a3b99-2496-4b77-8b40-cadf0d0e2f46" 
                          name="impl-test-api-subflow"/>

                <logger level="INFO" doc:name="End Main Process - /test/api" 
                        doc:id="1b868c79-9041-401f-86fd-5ef4c2b138ed" 
                        message='#[%dw 2.0
                        output application/json
                        ---
                        if (Mule::p("test.api.disable.payload.logs") default false) {} else

In [ ]:
"a".